In [ ]:
import random
import shutil
from pathlib import Path

random.seed(42)

DATASET1_DIR = Path(r"D:\Tabmed\dataset1")
MEDISEG_DIR = Path(r"D:\Tabmed\mediseg_yolo")
OUT_DIR = Path(r"D:\Tabmed\merged_dataset")

MEDISEG_SAMPLE_SIZE = 150

# 1. Пересоздаем структуру папок
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

for split in ["train", "val"]:
    (OUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# 2. Собираем dataset1 (разметка остается оригинальной: 0 - capsule, 1 - tablet)
d1_images = list((DATASET1_DIR / "images").glob("*.jpg")) + list((DATASET1_DIR / "images").glob("*.png"))
d1_pairs = []
for img in d1_images:
    lbl = DATASET1_DIR / "labels" / f"{img.stem}.txt"
    if lbl.is_file():
        d1_pairs.append((img, lbl, "dataset1"))

# 3. Собираем MEDISEG (случайные 150 снимков)
mediseg_imgs = list((MEDISEG_DIR / "images" / "train").glob("*.jpg")) + list((MEDISEG_DIR / "images" / "val").glob("*.jpg"))
random.shuffle(mediseg_imgs)
mediseg_sample = mediseg_imgs[:MEDISEG_SAMPLE_SIZE]

mediseg_pairs = []
for img in mediseg_sample:
    lbl_tr = MEDISEG_DIR / "labels" / "train" / f"{img.stem}.txt"
    lbl_vl = MEDISEG_DIR / "labels" / "val" / f"{img.stem}.txt"
    lbl = lbl_tr if lbl_tr.is_file() else lbl_vl
    if lbl.is_file():
        mediseg_pairs.append((img, lbl, "mediseg"))

# 4. Объединяем и делим на train (80%) и val (20%)
all_pairs = d1_pairs + mediseg_pairs
random.shuffle(all_pairs)

split_idx = int(len(all_pairs) * 0.8)
train_pairs = all_pairs[:split_idx]
val_pairs = all_pairs[split_idx:]

# 5. Копируем файлы с исправлением меток для MEDISEG
def copy_and_fix_labels(pairs, split):
    for img, lbl, source in pairs:
        shutil.copy2(img, OUT_DIR / "images" / split / img.name)
        dst_lbl_path = OUT_DIR / "labels" / split / lbl.name
        
        if source == "mediseg":
            # В MEDISEG были таблетки с меткой 0; меняем класс на 1 (tablet)
            lines = lbl.read_text(encoding="utf-8").strip().splitlines()
            fixed_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = "1"  # tablet
                    fixed_lines.append(" ".join(parts))
            with open(dst_lbl_path, "w", encoding="utf-8") as f:
                f.write("\n".join(fixed_lines) + ("\n" if fixed_lines else ""))
        else:
            # Для dataset1 копируем без изменений (0 - capsule, 1 - tablet)
            shutil.copy2(lbl, dst_lbl_path)

copy_and_fix_labels(train_pairs, "train")
copy_and_fix_labels(val_pairs, "val")

# 6. Записываем правильный data.yaml
yaml_text = f"""path: {OUT_DIR.as_posix()}
train: images/train
val: images/val

names:
  0: capsule
  1: tablet
"""
with open(OUT_DIR / "data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_text)

print(f"Датасет пересобран с корректными классами: {OUT_DIR}")
print(f"Всего: {len(all_pairs)} | Train: {len(train_pairs)} | Val: {len(val_pairs)}")

In [ ]:
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd()
PRETRAINED_WEIGHTS = BASE_DIR / "best.pt"
DATA_YAML = Path(r"D:\Tabmed\merged_dataset\data.yaml")

print(f"Базовые веса: {PRETRAINED_WEIGHTS}")
print(f"Датасет: {DATA_YAML}")

model = YOLO(str(PRETRAINED_WEIGHTS))

results = model.train(
    data=str(DATA_YAML),
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,           # RTX 4060 Ti
    workers=2,
    lr0=0.003,
    lrf=0.01,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    project=str(BASE_DIR / "runs" / "detect"),
    name="train_merged_v4",
    exist_ok=True
)

print("\nОбучение завершено! Чекпоинт сохранен: runs/detect/train_merged_v4/weights/best.pt")

In [ ]:
import random
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Загружаем исправленную модель v4
WEIGHTS_PATH = Path(r"D:\Tabmed\Tabmed_ml\runs\detect\train_merged_v4\weights\best.pt")
model = YOLO(str(WEIGHTS_PATH))

DIR_DATASET1 = Path(r"D:\Tabmed\dataset1\images")
DIR_MEDISEG = Path(r"D:\Tabmed\mediseg_yolo\images\val")

# Фиксируем seed для воспроизводимости выборки
random.seed(101)

imgs_d1 = list(DIR_DATASET1.glob("*.jpg")) + list(DIR_DATASET1.glob("*.png"))
imgs_med = list(DIR_MEDISEG.glob("*.jpg")) + list(DIR_MEDISEG.glob("*.png"))

# Берем по 10 снимков из каждого источника (всего 20)
sample_d1 = random.sample(imgs_d1, min(10, len(imgs_d1)))
sample_med = random.sample(imgs_med, min(10, len(imgs_med)))

test_images = [("Dataset1", p) for p in sample_d1] + [("MEDISEG", p) for p in sample_med]

print(f"{'№':<3} | {'Источник':<9} | {'Имя файла':<26} | {'Capsules':<9} | {'Tablets':<8} | {'Всего'}")
print("-" * 65)

# 2. Функция инференса и подготовки графиков
def process_image(img_path: Path):
    res = model.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    
    # Делаем рамки и подписи заметными (толщина линии 3, размер шрифта 1.2)
    annotated = res.plot(line_width=3, font_size=1.2)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    
    # Сжимаем превью, чтобы VS Code рендерил легко и без задержек
    h, w = annotated_rgb.shape[:2]
    scale = 500 / max(h, w)
    resized = cv2.resize(annotated_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    
    counts = {"capsule": 0, "tablet": 0}
    for b in res.boxes:
        cls_name = res.names[int(b.cls[0].item())]
        counts[cls_name] = counts.get(cls_name, 0) + 1
        
    return resized, counts

processed_data = []

for idx, (source, p) in enumerate(test_images, 1):
    img_rgb, counts = process_image(p)
    total = counts['capsule'] + counts['tablet']
    print(f"{idx:<3} | {source:<9} | {p.name:<26} | {counts['capsule']:<9} | {counts['tablet']:<8} | {total}")
    processed_data.append((source, p.name, img_rgb, counts))

# 3. Отрисовка сетки 5 строк x 4 колонки
fig, axes = plt.subplots(nrows=5, ncols=4, figsize=(18, 22))
axes = axes.flatten()

for i, (source, name, img_rgb, counts) in enumerate(processed_data):
    axes[i].imshow(img_rgb)
    summary = f"caps: {counts['capsule']}, tab: {counts['tablet']}"
    axes[i].set_title(f"[{source}] {name[:18]}..\n{summary}", fontsize=10)
    axes[i].axis("off")

plt.tight_layout()
plt.show()